In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.shape

(50000, 2)

In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

## Pre-Processing

In [8]:
## converting into a Lower Case

df['review'] = df['review'].str.lower()

In [9]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [10]:
## remove url

import re

def removal_url(text):
    text = re.sub(r"http\S+", "", text)
    return text
    
df['review'] = df['review'].apply(removal_url)

In [12]:
df.head(10)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive
5,"probably my all-time favorite movie, a story o...",positive
6,i sure would like to see a resurrection of a u...,positive
7,"this show was an amazing, fresh & innovative i...",negative
8,encouraged by the positive comments about this...,negative
9,if you like original gut wrenching laughter yo...,positive


In [13]:
def removal_punctuation(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text

df['review'] = df['review'].apply(removal_punctuation)

In [15]:
df.head(10)

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive
5,probably my alltime favorite movie a story of ...,positive
6,i sure would like to see a resurrection of a u...,positive
7,this show was an amazing fresh innovative ide...,negative
8,encouraged by the positive comments about this...,negative
9,if you like original gut wrenching laughter yo...,positive


In [16]:
def removal_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df['review'] = df['review'].apply(removal_html)

In [17]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [23]:
!pip install nltk

In [24]:
## Removing the stropwords

import nltk

nltk.download('puntk')
nltk.download('puntk_tab')
nltk.download('stopwords')

[nltk_data] Error loading puntk: Package 'puntk' not found in index
[nltk_data] Error loading puntk_tab: Package 'puntk_tab' not found in
[nltk_data]     index
[nltk_data] Downloading package stopwords to C:\Users\Mohd
[nltk_data]     Zaid\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [18]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [19]:
def removal_stopwords(text):
    tokens = word_tokenize(text)
    stop_word = stopwords.words('english')

    filtered_words = []

    for word in tokens:
        if word not in stop_word:
            filtered_words.append(word)

    return " ".join(filtered_words)

df['review'] = df['review'].apply(removal_stopwords)

In [20]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production br br filming tech...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


In [21]:
from nltk.stem import PorterStemmer

def removal_steming(text):
    ps = PorterStemmer()
    tokens = word_tokenize(text)
    stemmed_word = []

    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_word.append(stemmed_token)

    return " ".join(stemmed_word)

df['review'] = df['review'].apply(removal_steming)

In [22]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product br br film techniqu unass...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


## Encoding 

In [23]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['sentiment'] = le.fit_transform(df['sentiment'])

In [24]:
y = df['sentiment']

In [25]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

## Vectorization

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df['review'])

In [30]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4108217 stored elements and shape (49582, 5000)>

## Dataset & DataLoader

In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42
)

In [32]:
X_test.shape

(9917, 5000)

In [33]:
X_train.shape

(39665, 5000)

In [37]:
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

In [39]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [40]:
X_train

array([[0.23832435, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [42]:
train_set = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train.values, dtype=torch.float32)
)

test_set = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test.values, dtype=torch.float32)
)

In [43]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=True)

## Build Our RNN

In [44]:
import torch.optim as optim

In [52]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        ## Rnn Layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        ## fully connected Layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        ## Optional ==> shape (num of layers, batch_size, hidden_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out,_ = self.rnn(x, h0)
        ## 1st value = hidden state of all the timesteps => (batch, hidden, seq_len)
        ## 2nd Value = final hidden state of last timesteps

        out = self.fc(out[:, -1, :])

        return out 

In [53]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training The RNN

In [55]:
epochs = 10 

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => Probability

        loss = criterion(outputs, yb) # compute Loss
        loss.backward() # BackPropo
        optimizer.step() # Weight Update

    print(f"epoch {epoch+1}/{epochs} & Loss => {loss.item()}")

epoch 1/10 & Loss => 0.3020722270011902
epoch 2/10 & Loss => 0.10235217958688736
epoch 3/10 & Loss => 0.34341686964035034
epoch 4/10 & Loss => 0.1961195468902588
epoch 5/10 & Loss => 0.25444296002388
epoch 6/10 & Loss => 0.2938203811645508
epoch 7/10 & Loss => 0.5615663528442383
epoch 8/10 & Loss => 0.2317388504743576
epoch 9/10 & Loss => 0.08395855873823166
epoch 10/10 & Loss => 0.42078056931495667


In [58]:
# Evaluate 

model.eval()

with torch.no_grad():
    correct_val = 0
    total_val = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total_val += yb.size(0)
        correct_val += (predicted == yb).sum().item()

    print(f"accuracy => {correct_val / total_val * 100}")

accuracy => 87.32479580518302
